1/ Import data from github shared repository

In [1]:
#Import packages

import pandas as pd
import statsmodels.api as sm

In [2]:
#Load the dataset from GitHub

#USe CPI dates from 1958 to Febraury 1972
CPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/CPI_announcement_date.csv"
CPI_announcement_dates = pd.read_csv(CPI_announcement_dates)

#Use PPI dates from 1972 to 2024
PPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/PPI_announcement_date.csv"
PPI_announcement_dates = pd.read_csv(PPI_announcement_dates)

#FOMC scheduled interest rate announcement dates from 1990 to 2024
FOMC_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/FOMC_announcement_date.csv"
FOMC_announcement_dates = pd.read_csv(FOMC_announcement_dates, nrows=106)

#Fama_French_25_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Fama_French_25_portfolio_daily ="https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/25_Portfolios_Size_BM_Daily.csv"
Fama_French_25_portfolio_daily_EV = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=18, nrows=25920-19)
Fama_French_25_portfolio_daily_VW = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=25923, nrows=51825-25924)

#Ten_industry_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Ten_industry_portfolio_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/10_Industry_Portfolios_Daily.csv"
Ten_industry_portfolio_daily_EV = pd.read_csv(Ten_industry_portfolio_daily, skiprows=9, nrows=25911 - 10)
Ten_industry_portfolio_daily_VW = pd.read_csv(Ten_industry_portfolio_daily, skiprows=25914, nrows=51816 - 25915)

#Fama_French_3_factor_daily
Fama_French_3_factor_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/F-F_3_Factors_Daily.CSV"
Fama_French_3_factor_daily = pd.read_csv(Fama_French_3_factor_daily, skiprows=4, nrows=25906 - 5)

In [3]:
#Convert date columns to datetime format
CPI_announcement_dates = CPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
CPI_announcement_dates['A_date'] = pd.to_datetime(CPI_announcement_dates['A_date']).dt.date

PPI_announcement_dates = PPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
PPI_announcement_dates['A_date'] = pd.to_datetime(PPI_announcement_dates['A_date']).dt.date

FOMC_announcement_dates = FOMC_announcement_dates.rename(columns = {'FOMC Meeting Date': 'A_date'}).reset_index(drop=True)
FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date

Fama_French_25_portfolio_daily_EV = Fama_French_25_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_EV['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_25_portfolio_daily_VW = Fama_French_25_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_VW['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_EV = Ten_industry_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_EV['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_VW = Ten_industry_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_VW['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_3_factor_daily = Fama_French_3_factor_daily.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_3_factor_daily['Date'] = pd.to_datetime(Fama_French_3_factor_daily['Date'].astype(str), format='%Y%m%d').dt.date

C:\Users\bazel\AppData\Local\Temp\ipykernel_3552\1443429174.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date


2/ Split data between announcement days and non-announcement days for each portfolio

In [4]:
#Select CPI_announcement_dates up until February 1972
CPI_announcement_dates = CPI_announcement_dates.loc[CPI_announcement_dates['A_date'] <= pd.to_datetime('1972-02-01').date()]
CPI_announcement_dates = CPI_announcement_dates.reset_index(drop=True)

#Select PPI_announcement_dates from February 1972
PPI_announcement_dates = PPI_announcement_dates.loc[PPI_announcement_dates['A_date'] >= pd.to_datetime('1972-02-01').date()]
PPI_announcement_dates = PPI_announcement_dates.reset_index(drop=True)

In [5]:
all_A_dates = pd.concat([
    CPI_announcement_dates['A_date'],
    FOMC_announcement_dates['A_date'],
    PPI_announcement_dates['A_date']
])

Fama_French_25_portfolio_daily_EV_A_Day = Fama_French_25_portfolio_daily_EV[Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_A_Day = Fama_French_25_portfolio_daily_VW[Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_A_Day = Ten_industry_portfolio_daily_VW[Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_A_Day = Ten_industry_portfolio_daily_EV[Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)

Fama_French_25_portfolio_daily_EV_N_Day = Fama_French_25_portfolio_daily_EV[~Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_N_Day = Fama_French_25_portfolio_daily_VW[~Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_N_Day = Ten_industry_portfolio_daily_VW[~Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_N_Day = Ten_industry_portfolio_daily_EV[~Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)


In [6]:
#add the daily market Equity Risk premium and risk-free rate to all the portfolios
df_names = [
    'Fama_French_25_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_EV_N_Day',
    'Fama_French_25_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_EV_N_Day'
]

ff3_cols = ['Date', 'Mkt-RF', 'RF']

for name in df_names:
    globals()[name] = globals()[name].merge(Fama_French_3_factor_daily[ff3_cols], on='Date', how='left')

3/ Estimate unconditonal full sample beta

In [17]:
#We estimate the full sample market beta, on all dates (unconditonnaly to a-date and n-date), for each portfolio
df_names = [
    'Fama_French_25_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_EV'
]

regression_results = {}

for name in df_names:

    df = globals()[name]
    if name not in regression_results:
            regression_results[name] = {}

    print(f"\n=== Running regressions on: {name} ===")
    
    for col in df.columns:
        if col in ['Date', 'Mkt-RF', 'RF']:
            continue

        print(f"\n Regression for: {col}\n")

        # Dependent variable: portfolio excess return
        y = df[col] - df['RF']
        
        # Independent variable: market excess return
        X = sm.add_constant(df['Mkt-RF'])

        # Run OLS regression
        model = sm.OLS(y, X).fit()

        alpha = model.params['const']
        beta = model.params['Mkt-RF']

        regression_results[name][col] = {
            'alpha': model.params['const'],
            'beta': model.params['Mkt-RF']
        }



=== Running regressions on: Fama_French_25_portfolio_daily_EV ===

 Regression for: SMALL LoBM


 Regression for: ME1 BM2


 Regression for: ME1 BM3


 Regression for: ME1 BM4


 Regression for: SMALL HiBM


 Regression for: ME2 BM1


 Regression for: ME2 BM2


 Regression for: ME2 BM3


 Regression for: ME2 BM4


 Regression for: ME2 BM5


 Regression for: ME3 BM1


 Regression for: ME3 BM2


 Regression for: ME3 BM3


 Regression for: ME3 BM4


 Regression for: ME3 BM5


 Regression for: ME4 BM1


 Regression for: ME4 BM2


 Regression for: ME4 BM3


 Regression for: ME4 BM4


 Regression for: ME4 BM5


 Regression for: BIG LoBM


 Regression for: ME5 BM2


 Regression for: ME5 BM3


 Regression for: ME5 BM4


 Regression for: BIG HiBM


=== Running regressions on: Fama_French_25_portfolio_daily_VW ===

 Regression for: SMALL LoBM


 Regression for: ME1 BM2


 Regression for: ME1 BM3


 Regression for: ME1 BM4


 Regression for: SMALL HiBM


 Regression for: ME2 BM1


 Regression fo

In [18]:
print(regression_results)

{'Fama_French_25_portfolio_daily_EV': {'SMALL LoBM': {'alpha': 0.006071212432700728, 'beta': 1.0334281153878193}, 'ME1 BM2': {'alpha': 0.00041359464147141243, 'beta': 0.9587907745720557}, 'ME1 BM3': {'alpha': 0.011031569755227334, 'beta': 0.9377955097750711}, 'ME1 BM4': {'alpha': 0.019355338332586753, 'beta': 0.8766684505157636}, 'SMALL HiBM': {'alpha': 0.02529811939603908, 'beta': 0.8839617933560963}, 'ME2 BM1': {'alpha': -0.004833869162773327, 'beta': 1.0072126923033193}, 'ME2 BM2': {'alpha': 0.009235392466978453, 'beta': 0.9525217749395473}, 'ME2 BM3': {'alpha': 0.011541476482960496, 'beta': 0.9236562560533957}, 'ME2 BM4': {'alpha': 0.013936412942851181, 'beta': 0.9472009987863961}, 'ME2 BM5': {'alpha': 0.018409199963660576, 'beta': 1.0876726100629348}, 'ME3 BM1': {'alpha': -0.0024705997232164715, 'beta': 1.027984575688591}, 'ME3 BM2': {'alpha': 0.009224932085121866, 'beta': 0.941543914970137}, 'ME3 BM3': {'alpha': 0.010707285848540025, 'beta': 0.9342777775162481}, 'ME3 BM4': {'alph

In [ ]:
#Compute daily average excess returns for each portfolio

average_excess_returns = {
    'a_days': {},
    'n_days': {}
}

# List of portfolios to process
portfolio_df_names = {
    'a_days': Fama_French_25_portfolio_daily_EV_A_Day,
    'n_days': Fama_French_25_portfolio_daily_EV_N_Day
}

for period, df in portfolio_df_names.items():
    for col in df.columns:
        if col in ['Date', 'Mkt-RF', 'RF']:
            continue

        excess_returns = df[col] - df['RF']
        avg_excess = excess_returns.mean()

        average_excess_returns[period][col] = avg_excess

0.04618026284157385